# 03b — Feature engineering : séquences (FD001)

**Objectif** : transformer les données préparées en séquences de 30 cycles consécutifs par capteur — un format qui garde l'ordre des mesures dans le temps, plutôt que de les résumer en statistiques.

In [1]:
# --- Imports et configuration ---
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

from src.data.loaders import SENSOR_COLUMNS

SEED = 42
np.random.seed(SEED)

SUBSET = "FD001"
WINDOW = 30

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "dl"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
df_train = pd.read_parquet(INTERIM_DIR / f"train_{SUBSET}_prepared.parquet")
df_test = pd.read_parquet(INTERIM_DIR / f"test_{SUBSET}_prepared.parquet")
df_rul_test = pd.read_csv(INTERIM_DIR / f"rul_test_{SUBSET}_prepared.csv")

capteurs = [c for c in SENSOR_COLUMNS if c in df_train.columns]
print(f"{len(capteurs)} capteurs disponibles")
print(f"train : {df_train.shape}, test : {df_test.shape}")

15 capteurs disponibles
train : (20631, 20), test : (13096, 19)


## 1. Construction des séquences (train)

Pour chaque cycle à partir du 30e d'un moteur, on prend les 30 cycles précédents comme séquence, avec la RUL du dernier cycle comme cible. Les 29 premiers cycles de chaque moteur ne permettent pas de former une séquence complète et sont laissés de côté.

In [3]:
def construire_sequences(df, colonnes, window=WINDOW):
    X, y, unit_ids = [], [], []
    for unit_id, groupe in df.sort_values(["unit_number", "time_in_cycles"]).groupby("unit_number"):
        valeurs = groupe[colonnes].values
        cibles = groupe["RUL"].values
        for fin in range(window, len(groupe) + 1):
            X.append(valeurs[fin - window:fin])
            y.append(cibles[fin - 1])
            unit_ids.append(unit_id)
    return np.array(X), np.array(y), np.array(unit_ids)


X_train, y_train, unit_ids_train = construire_sequences(df_train, capteurs)

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)

X_train : (17731, 30, 15)
y_train : (17731,)


**Interprétation** : on obtient 17731 séquences, contre 20631 cycles au départ — la différence (2900) correspond exactement aux 29 premiers cycles écartés pour chacun des 100 moteurs, faute de 30 cycles complets disponibles. Chaque séquence contient 30 cycles sur 15 capteurs.

## 2. Construction des séquences (test)

Comme pour l'approche par statistiques, on ne garde qu'une seule séquence par moteur test — les 30 derniers cycles connus — le seul point pour lequel on a une RUL de référence.

In [4]:
longueurs_test = df_test.groupby("unit_number")["time_in_cycles"].max()
assert longueurs_test.min() >= WINDOW, "Au moins un moteur test a moins de 30 cycles enregistres."
print(f"Plus courte trajectoire test : {longueurs_test.min()} cycles (>= {WINDOW}, ok)")


def derniere_sequence(df, colonnes, window=WINDOW):
    X, unit_ids = [], []
    for unit_id, groupe in df.sort_values(["unit_number", "time_in_cycles"]).groupby("unit_number"):
        X.append(groupe[colonnes].values[-window:])
        unit_ids.append(unit_id)
    return np.array(X), np.array(unit_ids)


X_test, unit_ids_test = derniere_sequence(df_test, capteurs)
y_test = df_rul_test.set_index("unit_number").loc[unit_ids_test, "RUL_finale"].values

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

Plus courte trajectoire test : 31 cycles (>= 30, ok)
X_test : (100, 30, 15)
y_test : (100,)


**Interprétation** : la vérification confirme qu'aucun moteur test n'a moins de 30 cycles enregistrés (minimum 31) — on peut donc toujours extraire une séquence complète sans avoir besoin de compléter artificiellement des cycles manquants.

## 3. Sauvegarde

Les séquences sont sauvegardées au format numpy dans `data/processed/dl/` (le format tabulaire ne convient pas à des tableaux à 3 dimensions).

In [5]:
np.save(PROCESSED_DIR / f"X_train_{SUBSET}.npy", X_train)
np.save(PROCESSED_DIR / f"y_train_{SUBSET}.npy", y_train)
np.save(PROCESSED_DIR / f"unit_ids_train_{SUBSET}.npy", unit_ids_train)
np.save(PROCESSED_DIR / f"X_test_{SUBSET}.npy", X_test)
np.save(PROCESSED_DIR / f"y_test_{SUBSET}.npy", y_test)

print("Fichiers sauvegardes dans", PROCESSED_DIR)
for f in sorted(PROCESSED_DIR.glob(f"*{SUBSET}*")):
    print(" -", f.name)

Fichiers sauvegardes dans C:\cmapss-prediction-rul\data\processed\dl
 - unit_ids_train_FD001.npy
 - X_test_FD001.npy
 - X_train_FD001.npy
 - y_test_FD001.npy
 - y_train_FD001.npy


## Synthèse

Cette étape a produit des séquences de 30 cycles par capteur : plusieurs séquences par moteur pour le train, une seule (la plus récente) par moteur pour le test.